# Analyzing Zebrafish Social Interactions

In this notebook, we will perform quantitative analysis of the tracked trajectories, which computes interfish distance and approach velocity for both fish.

Prepared for DNC2026

> To export the notebook to PDF from a terminal, run `jupyter nbconvert --to pdf 2fish_analysis.ipynb`.

In [ ]:
from pathlib import Path
import csv
import gzip

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linewidth': 0.8,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.frameon': False,
})

def find_project_root(start_dir):
    start_dir = Path(start_dir).resolve()
    for candidate in (start_dir, *start_dir.parents):
        tracks_dir = candidate / 'tracks_wt'
        metrics_dir = candidate / 'precomputed_full_recording_metrics'
        if tracks_dir.exists() and metrics_dir.exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root containing tracks_wt and precomputed_full_recording_metrics.')

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)

file_path = PROJECT_ROOT / 'tracks_wt' / 'Zebrafish20250204_0944-pc2.csv'
start_frame = 0
end_frame = 100
include_other_fish = True
body_part = 'pec' # tail or head or pec
fps = 140
USE_PRECOMPUTED_TIMESERIES = True
PRECOMPUTED_METRICS_DIR = PROJECT_ROOT / 'precomputed_full_recording_metrics'
PRECOMPUTED_PAIR = 'fish1__fish2'
USE_SLIDING_TIMESERIES_AVERAGE = False
SLIDING_WINDOW_SECONDS = 2
SLIDING_STEP_SECONDS = 1
SAVE_SELECTED_COORDINATES_CSV = True
DISTANCE_Y_LIMITS = (0, 15)
APPROACH_VELOCITY_Y_LIMITS = (-20, 20)

def preview_csv(file_path, rows=5):
    with file_path.open(mode='r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        for index, row in enumerate(reader):
            print(row)
            if index + 1 >= rows:
                break

print(f'Using file: {file_path}')
print(f'Frame range: {start_frame} to {end_frame}')
print(f'Body part: {body_part}')
print(f'Use precomputed timeseries: {USE_PRECOMPUTED_TIMESERIES}')
print(f'Use sliding timeseries average: {USE_SLIDING_TIMESERIES_AVERAGE}')
print(f'Save selected coordinates CSV: {SAVE_SELECTED_COORDINATES_CSV}')
print(f'Interfish distance y-limits: {DISTANCE_Y_LIMITS}')
print(f'Approach velocity y-limits: {APPROACH_VELOCITY_Y_LIMITS}')
print(f'Sliding window: {SLIDING_WINDOW_SECONDS} s, step: {SLIDING_STEP_SECONDS} s')
preview_csv(file_path)

In [ ]:
def load_track_data(file_path, start_frame=0, end_frame=None, include_other_fish=False, body_part=body_part):
    track_data = {}
    missing_value_count = 0
    start_frame = max(start_frame, 0)
    end_frame = None if end_frame is None else max(end_frame, start_frame)

    with file_path.open(mode='r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        header = next(reader)
        available_fish = sorted(
            {column.split('_')[0] for column in header[1:] if column.endswith(f'{body_part}_x')}
        )
        if not available_fish:
            raise ValueError(f'No fish columns found for body part: {body_part}')

        fish_to_plot = available_fish if include_other_fish else [available_fish[0]]
        column_indices = {
            fish_name: (
                header.index(f'{fish_name}_{body_part}_x'),
                header.index(f'{fish_name}_{body_part}_y'),
                header.index(f'{fish_name}_{body_part}_z'),
            )
            for fish_name in fish_to_plot
        }
        track_data = {fish_name: {'x': [], 'y': [], 'z': []} for fish_name in fish_to_plot}

        for frame_index, row in enumerate(reader):
            if frame_index < start_frame:
                continue
            if end_frame is not None and frame_index > end_frame:
                break
            for fish_name in fish_to_plot:
                x_idx, y_idx, z_idx = column_indices[fish_name]
                for axis_name, axis_idx in (('x', x_idx), ('y', y_idx), ('z', z_idx)):
                    raw_value = row[axis_idx].strip()
                    if raw_value == '':
                        track_data[fish_name][axis_name].append(np.nan)
                        missing_value_count += 1
                    else:
                        track_data[fish_name][axis_name].append(float(raw_value))

    for fish_name, coords in track_data.items():
        track_data[fish_name] = {
            axis: np.asarray(values, dtype=float)
            for axis, values in coords.items()
        }

    return track_data, missing_value_count


def save_selected_coordinates_csv(track_data, file_path, start_frame, fps, body_part):
    end_frame = start_frame + len(next(iter(track_data.values()))['x']) - 1
    output_path = file_path.with_name(
        f'{file_path.stem}_{body_part}_frames_{start_frame}_{end_frame}_coordinates.csv'
    )
    with output_path.open(mode='w', newline='') as csvfile:
        writer = csv.DictWriter(
            csvfile,
            fieldnames=['frame_index', 'time_seconds', 'fish_name', 'x_cm', 'y_cm', 'z_cm'],
        )
        writer.writeheader()
        for fish_name, coords in track_data.items():
            for offset, (x_value, y_value, z_value) in enumerate(zip(coords['x'], coords['y'], coords['z'])):
                frame_index = start_frame + offset
                writer.writerow({
                    'frame_index': frame_index,
                    'time_seconds': frame_index / fps if fps else float(frame_index),
                    'fish_name': fish_name,
                    'x_cm': x_value,
                    'y_cm': y_value,
                    'z_cm': z_value,
                })
    return output_path


def stream_gzip_csv_rows(file_path, start_frame=None, end_frame=None):
    with gzip.open(file_path, mode='rt', newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            if start_frame is None and end_frame is None:
                yield row
                continue
            frame_index = int(row['frame_index'])
            if start_frame is not None and frame_index < start_frame:
                continue
            if end_frame is not None and frame_index > end_frame:
                break
            yield row


def read_first_gzip_csv_row(file_path):
    with gzip.open(file_path, mode='rt', newline='') as csvfile:
        return next(csv.DictReader(csvfile), None)


def get_precomputed_recording_dir(file_path, metrics_root):
    recording_dir = Path(metrics_root) / Path(file_path).stem
    if not recording_dir.exists():
        raise FileNotFoundError(f'No precomputed metrics found for {Path(file_path).stem} in {metrics_root}')
    return recording_dir


def load_precomputed_metadata(recording_dir):
    metadata_path = Path(recording_dir) / 'metadata.csv.gz'
    metadata = read_first_gzip_csv_row(metadata_path)
    if metadata is None:
        raise ValueError(f'No metadata rows found in {metadata_path}')
    return metadata


def compute_sliding_window_means(time_seconds, series_by_label, window_seconds, step_seconds):
    time_seconds = np.asarray(time_seconds, dtype=float)
    if time_seconds.size == 0:
        raise ValueError('No time-series samples available for sliding-window averaging.')
    if window_seconds <= 0 or step_seconds <= 0:
        raise ValueError('Sliding window and step must both be positive.')

    value_arrays = {
        label: np.asarray(values, dtype=float)
        for label, values in series_by_label.items()
    }

    start_time = float(time_seconds.min())
    max_time = float(time_seconds.max())
    centers = []
    averaged_values = {label: [] for label in value_arrays}

    current_start = start_time
    while current_start <= max_time:
        current_end = current_start + window_seconds
        mask = (time_seconds >= current_start) & (time_seconds <= current_end)
        if np.any(mask):
            centers.append(current_start + (window_seconds / 2.0))
            for label, values in value_arrays.items():
                window_values = values[mask]
                finite_values = window_values[np.isfinite(window_values)]
                averaged_values[label].append(float(np.mean(finite_values)) if finite_values.size else np.nan)
        current_start += step_seconds

    return np.asarray(centers, dtype=float), {
        label: np.asarray(values, dtype=float)
        for label, values in averaged_values.items()
    }


def maybe_apply_sliding_window(time_seconds, series_by_label):
    if not USE_SLIDING_TIMESERIES_AVERAGE:
        return np.asarray(time_seconds, dtype=float), {
            label: np.asarray(values, dtype=float)
            for label, values in series_by_label.items()
        }
    return compute_sliding_window_means(
        time_seconds,
        series_by_label,
        SLIDING_WINDOW_SECONDS,
        SLIDING_STEP_SECONDS,
    )


track_data, missing_value_count = load_track_data(
    file_path=file_path,
    start_frame=start_frame,
    end_frame=end_frame,
    include_other_fish=include_other_fish,
    body_part=body_part,
)
fish_names = list(track_data.keys())
timeseries_fish_names = fish_names.copy()
precomputed_recording_dir = None
precomputed_metadata = None

if USE_PRECOMPUTED_TIMESERIES:
    precomputed_recording_dir = get_precomputed_recording_dir(file_path, PRECOMPUTED_METRICS_DIR)
    precomputed_metadata = load_precomputed_metadata(precomputed_recording_dir)
    metadata_fish_names = [name for name in precomputed_metadata.get('fish_names', '').split(';') if name]
    if metadata_fish_names:
        timeseries_fish_names = metadata_fish_names
    precomputed_body_part = precomputed_metadata.get('body_part')
    precomputed_fps = float(precomputed_metadata['fps']) if precomputed_metadata.get('fps') else None
    print(f'Using precomputed metrics from: {precomputed_recording_dir}')
    if precomputed_body_part and precomputed_body_part != body_part:
        print(f'Warning: notebook body_part={body_part} differs from precomputed body_part={precomputed_body_part}.')
    if precomputed_fps is not None and not np.isclose(precomputed_fps, fps):
        print(f'Warning: notebook fps={fps} differs from precomputed fps={precomputed_fps}.')

frame_count = len(next(iter(track_data.values()))['x'])
print(f'Loaded {frame_count} frames for {len(fish_names)} fish: {fish_names}')
print(f'Converted {missing_value_count} blank coordinate values to NaN.')
if SAVE_SELECTED_COORDINATES_CSV:
    coordinate_csv_path = save_selected_coordinates_csv(
        track_data=track_data,
        file_path=file_path,
        start_frame=start_frame,
        fps=fps,
        body_part=body_part,
    )
    print(f'Saved selected coordinates to {coordinate_csv_path}')

## Interfish Distance Analysis

This section quantifies pairwise distance across the selected frame window. The plot below shows how the inter-fish distance evolves over time for the loaded trajectory segment.

In [ ]:
def compute_distance(track_data, fish1, fish2):
    position_1 = np.column_stack((track_data[fish1]['x'], track_data[fish1]['y'], track_data[fish1]['z']))
    position_2 = np.column_stack((track_data[fish2]['x'], track_data[fish2]['y'], track_data[fish2]['z']))
    return np.linalg.norm(position_1 - position_2, axis=1)


def load_precomputed_distance(file_path, metrics_root, pair, start_frame=0, end_frame=None):
    recording_dir = get_precomputed_recording_dir(file_path, metrics_root)
    distance_path = recording_dir / f'{pair}_distance.csv.gz'
    rows = list(stream_gzip_csv_rows(distance_path, start_frame=start_frame, end_frame=end_frame))
    if not rows:
        raise ValueError('No precomputed distance samples found in the selected frame window.')
    time_seconds = np.asarray([float(row['time_seconds']) for row in rows], dtype=float)
    time_seconds = time_seconds - time_seconds[0]
    distances = np.asarray([float(row['distance_cm']) for row in rows], dtype=float)
    return time_seconds, distances


def plot_distance_series(time_seconds, distances, label, output_path=None, y_limits=None):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(time_seconds, distances, label=label, color='royalblue', linewidth=2)
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('Interfish Distance [cm]')
    ax.legend(loc='best', frameon=True)
    ax.grid(visible=False)
    if len(time_seconds):
        ax.set_xlim(0, time_seconds.max())
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


def plot_distance_over_time(track_data, fish1, fish2, fps=140, output_path=None, y_limits=None):
    distances = compute_distance(track_data, fish1, fish2)
    time_seconds = np.arange(len(distances)) / fps
    time_seconds, smoothed_series = maybe_apply_sliding_window(
        time_seconds,
        {f'{fish1} and {fish2}': distances},
    )
    plot_distance_series(
        time_seconds,
        smoothed_series[f'{fish1} and {fish2}'],
        label=f'{fish1} and {fish2}',
        output_path=output_path,
        y_limits=y_limits,
    )


if USE_PRECOMPUTED_TIMESERIES:
    if len(timeseries_fish_names) >= 2:
        output_path = file_path.with_name(f'{file_path.stem}_{body_part}_interfish_distance.png')
        time_seconds, distances = load_precomputed_distance(
            file_path=file_path,
            metrics_root=PRECOMPUTED_METRICS_DIR,
            pair=PRECOMPUTED_PAIR,
            start_frame=start_frame,
            end_frame=end_frame,
        )
        distance_label = f'{timeseries_fish_names[0]} and {timeseries_fish_names[1]}'
        time_seconds, smoothed_series = maybe_apply_sliding_window(
            time_seconds,
            {distance_label: distances},
        )
        plot_distance_series(
            time_seconds,
            smoothed_series[distance_label],
            label=distance_label,
            output_path=output_path,
            y_limits=DISTANCE_Y_LIMITS,
        )
        print(f'Saved interfish distance plot to {output_path}')
    else:
        print('Not enough fish names available to label precomputed distance.')
else:
    if len(fish_names) >= 2:
        output_path = file_path.with_name(f'{file_path.stem}_{body_part}_interfish_distance.png')
        plot_distance_over_time(
            track_data,
            fish_names[0],
            fish_names[1],
            fps=fps,
            output_path=output_path,
            y_limits=DISTANCE_Y_LIMITS,
        )
        print(f'Saved interfish distance plot to {output_path}')
    else:
        print('Not enough fish data to compute distance.')

## Approach Velocity

Approach velocity projects each fish's velocity onto the line connecting the pair, to describe motion toward or away from the other fish across the recording window.

In [ ]:
def compute_approach_velocity(track_data, fish1, fish2, fps=140):
    position_1 = np.column_stack((track_data[fish1]['x'], track_data[fish1]['y'], track_data[fish1]['z']))
    position_2 = np.column_stack((track_data[fish2]['x'], track_data[fish2]['y'], track_data[fish2]['z']))

    velocity_1 = np.gradient(position_1, axis=0) * fps
    velocity_2 = np.gradient(position_2, axis=0) * fps

    relative_position_12 = position_2 - position_1
    distance = np.linalg.norm(relative_position_12, axis=1)
    unit_vector_12 = np.divide(
        relative_position_12,
        distance[:, None],
        out=np.zeros_like(relative_position_12),
        where=distance[:, None] > 0,
    )
    unit_vector_21 = -unit_vector_12

    approach_velocity_1 = np.einsum('ij,ij->i', velocity_1, unit_vector_12)
    approach_velocity_2 = np.einsum('ij,ij->i', velocity_2, unit_vector_21)
    return approach_velocity_1, approach_velocity_2


def load_precomputed_approach_velocity(file_path, metrics_root, pair, start_frame=0, end_frame=None):
    recording_dir = get_precomputed_recording_dir(file_path, metrics_root)
    approach_path = recording_dir / f'{pair}_approach_velocity.csv.gz'
    rows = list(stream_gzip_csv_rows(approach_path, start_frame=start_frame, end_frame=end_frame))
    if not rows:
        raise ValueError('No precomputed approach-velocity samples found in the selected frame window.')
    time_seconds = np.asarray([float(row['time_seconds']) for row in rows], dtype=float)
    time_seconds = time_seconds - time_seconds[0]
    value_fields = [field for field in rows[0].keys() if field.startswith('approach_velocity_')]
    approach_series = {
        field.removeprefix('approach_velocity_').removesuffix('_cm_s'): np.asarray(
            [float(row[field]) for row in rows],
            dtype=float,
        )
        for field in value_fields
    }
    return time_seconds, approach_series


def plot_approach_velocity_series(time_seconds, approach_series, output_path=None, y_limits=None):
    fig, ax = plt.subplots(figsize=(11, 5.5))
    colors = ['royalblue', 'crimson', 'seagreen', 'darkorange']
    for index, (label, values) in enumerate(approach_series.items()):
        ax.plot(time_seconds, values, label=label, color=colors[index % len(colors)], linewidth=2)
    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('Approach velocity [cm/s]')
    ax.legend(title='Fish', loc='upper right', frameon=True)
    ax.grid(visible=False)
    if len(time_seconds):
        ax.set_xlim(0, time_seconds.max())
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


def plot_approach_velocity(track_data, fish1, fish2, fps=140, output_path=None, y_limits=None):
    approach_velocity_1, approach_velocity_2 = compute_approach_velocity(track_data, fish1, fish2, fps=fps)
    time_seconds = np.arange(len(approach_velocity_1)) / fps
    time_seconds, approach_series = maybe_apply_sliding_window(
        time_seconds,
        {
            fish1: approach_velocity_1,
            fish2: approach_velocity_2,
        },
    )
    plot_approach_velocity_series(
        time_seconds,
        approach_series,
        output_path=output_path,
        y_limits=y_limits,
    )


if USE_PRECOMPUTED_TIMESERIES:
    output_path = file_path.with_name(f'{file_path.stem}_{body_part}_approach_velocity.png')
    time_seconds, approach_series = load_precomputed_approach_velocity(
        file_path=file_path,
        metrics_root=PRECOMPUTED_METRICS_DIR,
        pair=PRECOMPUTED_PAIR,
        start_frame=start_frame,
        end_frame=end_frame,
    )
    time_seconds, approach_series = maybe_apply_sliding_window(time_seconds, approach_series)
    plot_approach_velocity_series(
        time_seconds,
        approach_series,
        output_path=output_path,
        y_limits=APPROACH_VELOCITY_Y_LIMITS,
    )
    print(f'Saved approach velocity plot to {output_path}')
else:
    if len(fish_names) >= 2:
        output_path = file_path.with_name(f'{file_path.stem}_{body_part}_approach_velocity.png')
        plot_approach_velocity(
            track_data,
            fish_names[0],
            fish_names[1],
            fps=fps,
            output_path=output_path,
            y_limits=APPROACH_VELOCITY_Y_LIMITS,
        )
        print(f'Saved approach velocity plot to {output_path}')
    else:
        print('Not enough fish data to compute approach velocity.')